In [1]:
import json
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")
all_dfs = []

for filepath in sorted(RAW_DIR.glob("*.json")):
    with open(filepath) as f:
        data = json.load(f)
    df = pd.DataFrame(data["hourly"])
    df["city_file"] = filepath.stem
    df["api_latitude"] = data["latitude"]
    df["api_longitude"] = data["longitude"]
    all_dfs.append(df)

combined = pd.concat(all_dfs, ignore_index=True)
combined["time"] = pd.to_datetime(combined["time"])
print(f"Total rows: {len(combined):,}")
combined.head()

Total rows: 1,680


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m,pressure_msl,weather_code,city_file,api_latitude,api_longitude
0,2026-05-03 00:00:00,18.5,82,0.0,6.6,22,1008.2,2,alexandria_20260503T004508Z,31.1875,29.9375
1,2026-05-03 01:00:00,18.2,86,0.0,5.0,21,1007.8,2,alexandria_20260503T004508Z,31.1875,29.9375
2,2026-05-03 02:00:00,18.0,88,0.0,3.7,11,1007.8,2,alexandria_20260503T004508Z,31.1875,29.9375
3,2026-05-03 03:00:00,18.1,86,0.0,4.8,312,1008.5,2,alexandria_20260503T004508Z,31.1875,29.9375
4,2026-05-03 04:00:00,18.2,82,0.0,7.1,315,1009.0,3,alexandria_20260503T004508Z,31.1875,29.9375


In [2]:
nulls = combined.isnull().sum()
nulls_pct = (nulls / len(combined) * 100).round(2)
quality = pd.DataFrame({"null_count": nulls, "null_pct": nulls_pct})
quality[quality["null_count"] > 0]

,null_count,null_pct


In [3]:
numeric_cols = combined.select_dtypes(include="number").columns
combined[numeric_cols].describe().T[["min", "max", "mean"]]

,min,max,mean
temperature_2m,-1.500000,35.000000,16.935655
relative_humidity_2m,18.000000,100.000000,66.438690
precipitation,0.000000,6.000000,0.059690
wind_speed_10m,0.300000,38.500000,12.109286
wind_direction_10m,1.000000,360.000000,183.857738
pressure_msl,993.300000,1028.100000,1014.431905
weather_code,0.000000,95.000000,9.313095
api_latitude,-33.989456,64.139565,18.104423
api_longitude,-73.993080,151.195510,30.058192


In [4]:
# Compare what you requested vs. what the API returned
requested = {
    "cairo":      (30.0444,  31.2357),
    "alexandria": (31.2001,  29.9187),
    "london":     (51.5074,  -0.1278),
    # ...
}

for city_file in combined["city_file"].unique():
    city_key = city_file.split("_")[0]
    if city_key in requested:
        req_lat, req_lon = requested[city_key]
        got = combined[combined["city_file"] == city_file].iloc[0]
        delta_lat = abs(req_lat - got["api_latitude"])
        delta_lon = abs(req_lon - got["api_longitude"])
        print(f"{city_key:12} requested ({req_lat:.4f}, {req_lon:.4f}) "
              f"got ({got['api_latitude']:.4f}, {got['api_longitude']:.4f}) "
              f"delta=({delta_lat:.4f}, {delta_lon:.4f})")

alexandria   requested (31.2001, 29.9187) got (31.1875, 29.9375) delta=(0.0126, 0.0188)
cairo        requested (30.0444, 31.2357) got (30.0625, 31.2500) delta=(0.0181, 0.0143)
london       requested (51.5074, -0.1278) got (51.5115, -0.1308) delta=(0.0041, 0.0030)


In [5]:
combined.groupby("city_file")["time"].agg(["min", "max", "count"])

,min,max,count
city_file,,,
alexandria_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
cairo_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
cape_town_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
london_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
mumbai_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
new_york_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
reykjavik_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
sao_paulo_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
sydney_20260503T004508Z,2026-05-03,2026-05-09 23:00:00,168
